In [ ]:
import pandas as pd
import os
import re

# Function to clean company names
def clean_company_name(company):
    """
    Cleans company names by:
    - Converting to lowercase
    - Removing common suffixes (e.g., "Co., Ltd.", "Ltd.", "Corporation", etc.)
    - Removing extra spaces and punctuation
    """
    company = str(company).strip().lower()

    patterns_to_remove = [
        r'\s*\.co\.,?\s*ltd\.?$',  
        r'\s*co\.,?\s*ltd\.?$',    
        r'\s*ltd[.]?$',            
        r'\s*corporation$',        
        r'\s*inc[.]?$',            
        r'\s*corp$',               
        r'\s*company$',            
        r'\s*\.?company$',         
        r'\s*,?company$',          
        r'\s*co[.]?$',             
        r'\s*,?$'        
    ]

    for pattern in patterns_to_remove:
        company = re.sub(pattern, '', company, flags=re.IGNORECASE).strip()

    return company

# File Paths
raw_data_path = "../data/raw/korean_stock_data.csv"
tickers_path = "../data/raw/kospi200_companies.csv"
output_path = "../data/interim/korean_stock_data_with_tickers.csv"

# Load Data
if not os.path.exists(raw_data_path):
    raise FileNotFoundError(f"⚠️ Raw data file not found: {raw_data_path}")

if not os.path.exists(tickers_path):
    raise FileNotFoundError(f"⚠️ Ticker file not found: {tickers_path}")

df_stock = pd.read_csv(raw_data_path)
df_tickers = pd.read_csv(tickers_path, dtype={"ticker": str})  # Ensure ticker is treated as a string

# Clean company names in both datasets
df_stock["company"] = df_stock["company"].apply(clean_company_name)
df_tickers["company"] = df_tickers["company"].apply(clean_company_name)

# Create a dictionary for fast lookup (company -> ticker)
ticker_dict = dict(zip(df_tickers["company"], df_tickers["ticker"]))

# Match tickers to companies
df_stock["ticker"] = df_stock["company"].map(ticker_dict)

# Sort: Move unmatched companies to the bottom
df_stock = df_stock.sort_values(by="ticker", ascending=False).reset_index(drop=True)

# Save the updated data
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_stock.to_csv(output_path, index=False)

print(f"✅ Ticker-matched data saved to: {output_path}")


✅ Ticker-matched data saved to: ../data/interim/korean_stock_data_with_tickers.csv


In [1]:
# Extract companies without a matched ticker
unmatched_companies = df_stock[df_stock["ticker"].isna()][["company"]].drop_duplicates()

# Print the unmatched companies
if unmatched_companies.empty:
    print("✅ All companies have matched tickers.")
else:
    print("⚠️ The following companies do not have a matched ticker:")
    print(unmatched_companies)

# Optional: Save unmatched companies for further investigation
unmatched_output_path = "../data/interim/unmatched_companies.csv"
unmatched_companies.to_csv(unmatched_output_path, index=False)
print(f"📂 Unmatched companies saved to: {unmatched_output_path}")


NameError: name 'df_stock' is not defined

# Clean EWI

In [ ]:
import os
import pandas as pd
import numpy as np

def clean_ewi_file(input_path="../data/processed/kospi_200_equal_weighted_index_final.csv",
                   output_path="../data/processed/kospi_200_equal_weighted_index_final_clean.csv",
                   method="drop"):
    """
    Clean the EWI file by replacing infinite values with NaN,
    then either dropping or filling the NaN values.
    
    Args:
        input_path (str): Path to the raw EWI file.
        output_path (str): Path to save the cleaned EWI file.
        method (str): "drop" to drop rows with NaNs, "fill" to forward/backward fill them.
    
    Returns:
        DataFrame: Cleaned EWI dataframe.
    """
    # Load the file
    df = pd.read_csv(input_path, parse_dates=["date"])
    # Replace infinite values with NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    if method == "drop":
        df_clean = df.dropna()
    elif method == "fill":
        df_clean = df.fillna(method="ffill").fillna(method="bfill")
    else:
        raise ValueError("Invalid method specified. Use 'drop' or 'fill'.")
    
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_clean.to_csv(output_path, index=False)
    print(f"Saved cleaned EWI data to {output_path}")
    return df_clean

# TODO: Need to clean the inf and -inf